# Transformer-Only Ablation V2 (one-hot + BCE)

This notebook is the controlled Transformer-only ablation of the completed
`train_all_twotower_v2.ipynb` experiment.

The **only scientific change** is the model architecture:

- keep the raw UCR time-series input;
- keep the time-series linear embedding, ReLU, and temporal Transformer;
- remove the fractional `Aout` input and fractional tower;
- remove token concatenation and the final fusion Transformer;
- connect the flattened temporal-Transformer representation directly to the
  same dropout/classification head.

Everything needed for a fair comparison is inherited from Two-Tower V2:

- the same cleaned UCR datasets and deterministic shared train/validation indices;
- per-sample time-series standardization and train-derived sequence length;
- one-hot/binary-column labels with `BCEWithLogitsLoss`;
- corrected binary ACC/AUC and the same multiclass metrics;
- Adam (`1e-4`), batch size 8, cosine scheduler, hidden dimension 16, two heads,
  two layers, seed 42, and 60 full training epochs;
- best-checkpoint selection by validation BCE loss;
- official TEST evaluation exactly once after restoring the best checkpoint;
- resumable, isolated output under `final_runs_v2/transformer_only/seed_42`.

`Aout` files are checked only to keep the discovered 125-dataset universe identical
to the full Two-Tower run. Their values are never loaded, scaled, or passed to the
Transformer-only model.

## First local run

1. Confirm `DATA_ROOT` and `OUTPUT_ROOT` in Cell 1.
2. Leave `RUN_SMOKE_TEST = True` and `RUN_FULL_EXPERIMENT = False`, then restart
   the kernel and run all cells.
3. Confirm the audit prints `uses_aout=False`, and Coffee/ArrowHead finish in the
   isolated `_debug/transformer_only/seed_42` directory.
4. Set `RUN_SMOKE_TEST = False`, `RUN_FULL_EXPERIMENT = True`, restart the kernel,
   and run all cells again for the final 125-dataset experiment.

In [ ]:
# Cell 1 - imports and experiment configuration
from __future__ import annotations

import copy
import json
import os
import platform
import random
import sys
import traceback
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.backends.cudnn as cudnn
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.preprocessing import label_binarize
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader, Dataset


# -----------------------------------------------------------------------------
# EDIT THESE TWO PATHS FOR YOUR WINDOWS MACHINE
# -----------------------------------------------------------------------------
DATA_ROOT = Path(r"D:\2025暑期科研\UCRArchive_2018\UCRArchive_2018")
OUTPUT_ROOT = Path(r"D:\2025暑期科研\UCRArchive_2018\TwoTower_Project_V2\final_runs_v2")


# Reproducible experiment settings: identical to the completed Two-Tower V2.
MODEL_NAME = "transformer_only"
EXPERIMENT_VERSION = "transformer_only_v2_bce_correct_metrics"
SEED = 42
BATCH_SIZE = 8
NUM_EPOCHS = 60
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 0.0
T_MAX = 50
ETA_MIN = 0.0
VAL_FRACTION = 0.20
MIN_EFFECTIVE_LENGTH = 8
NUM_WORKERS = 0

# Train all 60 epochs for equal-length Figure 2/3 histories. Final TEST metrics
# use the checkpoint with minimum validation loss, not necessarily epoch 60.
TRAIN_FULL_EPOCHS_FOR_CURVES = True

# Safe defaults: run the isolated smoke test before enabling the final run.
RUN_SMOKE_TEST = False
RUN_FULL_EXPERIMENT = True
SMOKE_DATASETS = ["Coffee", "ArrowHead"]  # binary + multiclass
SMOKE_EPOCHS = 3

# Resume skips datasets already present in this model's final_results.csv.
RESUME_COMPLETED_DATASETS = True
SAVE_INDIVIDUAL_CURVES = True

# Select GPU when available. Change to "cpu" only for debugging without CUDA.
DEVICE_REQUEST = "cuda:0"

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# Cell 2 - determinism, paths, atomic persistence, and run metadata
def set_global_seed(seed: int) -> None:
    """Reset all relevant RNGs. Called again at the start of every dataset."""
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    cudnn.deterministic = True
    cudnn.benchmark = False


set_global_seed(SEED)


def resolve_device(requested: str) -> torch.device:
    if requested.startswith("cuda") and torch.cuda.is_available():
        return torch.device(requested)
    return torch.device("cpu")


DEVICE = resolve_device(DEVICE_REQUEST)
print("Resolved device:", DEVICE)


def model_output_dir(debug: bool = False) -> Path:
    if debug:
        return OUTPUT_ROOT / "_debug" / MODEL_NAME / f"seed_{SEED}"
    return OUTPUT_ROOT / MODEL_NAME / f"seed_{SEED}"


def ensure_output_tree(base_dir: Path) -> dict[str, Path]:
    paths = {
        "base": base_dir,
        "checkpoints": base_dir / "checkpoints",
        "curves": base_dir / "curves",
        "dataset_histories": base_dir / "dataset_histories",
        "shared_splits": OUTPUT_ROOT / "_shared_splits" / f"seed_{SEED}",
    }
    for path in paths.values():
        path.mkdir(parents=True, exist_ok=True)
    return paths


def atomic_write_csv(df: pd.DataFrame, path: Path) -> None:
    """Write through a temporary file so an interrupted write keeps the old CSV."""
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    df.to_csv(tmp, index=False, encoding="utf-8")
    os.replace(tmp, path)


def atomic_write_json(payload: dict, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with open(tmp, "w", encoding="utf-8") as handle:
        json.dump(payload, handle, ensure_ascii=False, indent=2)
    os.replace(tmp, path)


def append_or_replace_dataset_rows(
    csv_path: Path,
    new_df: pd.DataFrame,
    dataset_name: str,
) -> pd.DataFrame:
    if csv_path.exists():
        old_df = pd.read_csv(csv_path)
        if "dataset" in old_df.columns:
            old_df = old_df[old_df["dataset"] != dataset_name]
        combined = pd.concat([old_df, new_df], ignore_index=True)
    else:
        combined = new_df.copy()
    sort_cols = [c for c in ["dataset", "epoch"] if c in combined.columns]
    if sort_cols:
        combined = combined.sort_values(sort_cols).reset_index(drop=True)
    atomic_write_csv(combined, csv_path)
    return combined


def base_run_config(debug: bool, epochs: int) -> dict:
    return {
        "experiment_version": EXPERIMENT_VERSION,
        "model": MODEL_NAME,
        "architecture": "time embedding -> temporal Transformer -> flatten -> dropout -> classifier",
        "ablation": "fractional/Aout tower and final fusion Transformer removed",
        "uses_aout": False,
        "seed": SEED,
        "debug": bool(debug),
        "data_root": str(DATA_ROOT),
        "output_root": str(OUTPUT_ROOT),
        "device_requested": DEVICE_REQUEST,
        "device_resolved": str(DEVICE),
        "batch_size": BATCH_SIZE,
        "num_epochs": int(epochs),
        "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "scheduler": "CosineAnnealingLR",
        "t_max": T_MAX,
        "eta_min": ETA_MIN,
        "val_fraction": VAL_FRACTION,
        "label_encoding": "one_hot_or_binary_column",
        "loss": "BCEWithLogitsLoss",
        "binary_prediction": "sigmoid(logit) >= 0.5",
        "multiclass_prediction": "argmax(logits)",
        "multiclass_auc": "macro one-vs-rest over valid classes",
        "checkpoint_selection": "minimum validation BCE loss",
        "test_evaluation": "once after restoring best checkpoint",
        "python": platform.python_version(),
        "pytorch": torch.__version__,
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "created_utc": datetime.now(timezone.utc).isoformat(),
    }

In [ ]:
# Cell 3 - preprocessing and deterministic shared train/validation split
def clean_and_pad_timeseries(
    raw_2d: np.ndarray,
    min_len: int = 8,
    cap_len: int | None = None,
    pad_value: float = 0.0,
    per_sample_standardize: bool = True,
    fixed_len: int | None = None,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Clean tail NaNs, optionally z-score each sample, then pad or truncate."""
    rows, keep_idx, real_lens = [], [], []

    for i, row in enumerate(raw_2d):
        valid_vals = row[~np.isnan(row)]
        length = int(valid_vals.shape[0])
        if length < min_len:
            continue
        if per_sample_standardize:
            mu = float(valid_vals.mean())
            sigma = float(valid_vals.std())
            valid_vals = (valid_vals - mu) / (sigma if sigma > 0 else 1.0)
        rows.append(valid_vals)
        keep_idx.append(i)
        real_lens.append(length)

    if not rows:
        raise ValueError("All samples were filtered out; lower min_len or inspect the data.")

    max_real_len = max(real_lens)
    if fixed_len is not None:
        target_len = int(fixed_len)
    elif cap_len is not None:
        target_len = min(max_real_len, int(cap_len))
    else:
        target_len = max_real_len

    padded = []
    for values in rows:
        if values.shape[0] >= target_len:
            values = values[:target_len]
        else:
            values = np.pad(
                values,
                (0, target_len - values.shape[0]),
                constant_values=pad_value,
            )
        padded.append(values)

    return (
        np.stack(padded, axis=0).astype("float32"),
        np.asarray(keep_idx, dtype=np.int64),
        np.asarray(real_lens, dtype=np.int64),
    )


def make_class_safe_split(
    labels: np.ndarray,
    dataset_name: str,
    split_dir: Path,
    val_fraction: float = 0.20,
    seed: int = 42,
) -> tuple[np.ndarray, np.ndarray, Path]:
    """
    Create or reload a deterministic split from the cleaned official TRAIN set.

    Each class keeps at least one sample in the optimization subset. Classes with
    only one sample remain entirely in training. The saved split is shared by all
    V2 models and validated before reuse.
    """
    split_dir.mkdir(parents=True, exist_ok=True)
    split_path = split_dir / f"{dataset_name}_split.npz"
    labels = np.asarray(labels)

    if split_path.exists():
        saved = np.load(split_path, allow_pickle=False)
        train_idx = saved["train_idx"].astype(np.int64)
        val_idx = saved["val_idx"].astype(np.int64)
        saved_n = int(saved["n_samples"])
        if saved_n != len(labels):
            raise ValueError(
                f"Saved split for {dataset_name} has n={saved_n}, "
                f"but current cleaned training data has n={len(labels)}. "
                "Delete only this split file after verifying preprocessing."
            )
        return train_idx, val_idx, split_path

    rng = np.random.default_rng(seed)
    train_parts, val_parts = [], []
    for class_value in np.unique(labels):
        class_idx = np.flatnonzero(labels == class_value)
        class_idx = rng.permutation(class_idx)
        if len(class_idx) <= 1:
            n_val = 0
        else:
            n_val = max(1, int(round(len(class_idx) * val_fraction)))
            n_val = min(n_val, len(class_idx) - 1)
        val_parts.append(class_idx[:n_val])
        train_parts.append(class_idx[n_val:])

    train_idx = np.sort(np.concatenate(train_parts)).astype(np.int64)
    nonempty_val_parts = [part for part in val_parts if len(part) > 0]
    if nonempty_val_parts:
        val_idx = np.sort(np.concatenate(nonempty_val_parts)).astype(np.int64)
    else:
        raise ValueError(
            f"{dataset_name} cannot create a non-empty class-safe validation split."
        )

    if set(train_idx).intersection(set(val_idx)):
        raise RuntimeError("Train/validation indices overlap.")
    if len(train_idx) + len(val_idx) != len(labels):
        raise RuntimeError("Train/validation split does not cover every cleaned sample.")

    np.savez_compressed(
        split_path,
        train_idx=train_idx,
        val_idx=val_idx,
        n_samples=np.asarray(len(labels), dtype=np.int64),
        seed=np.asarray(seed, dtype=np.int64),
    )
    return train_idx, val_idx, split_path


In [ ]:
# Cell 4 - controlled Transformer-only architecture and dataset wrapper
class TransformerOnly(nn.Module):
    """
    Controlled ablation of TwoTowerTransformer.

    Kept:
      [B, T, 1] -> linear embedding -> ReLU -> temporal Transformer
      -> flatten -> dropout -> logits

    Removed:
      fractional Aout input/tower, token concatenation, and final fusion Transformer.
    """

    def __init__(
        self,
        input_dim1: int,
        hidden_dim1: int,
        num_heads: int,
        num_layers: int,
        num_classes: int,
        seq_len1: int,
    ):
        super().__init__()
        assert hidden_dim1 % num_heads == 0

        self.embedding1 = nn.Linear(input_dim1, hidden_dim1)
        self.relu = nn.ReLU()
        self.transformer1 = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(
                d_model=hidden_dim1,
                nhead=num_heads,
                batch_first=True,
            ),
            num_layers=num_layers,
        )
        self.dropout = nn.Dropout(0.3)
        self.seq_len1 = int(seq_len1)
        self.fc = nn.Linear(hidden_dim1 * self.seq_len1, num_classes)

    def forward(self, x1: torch.Tensor) -> torch.Tensor:
        if x1.ndim == 2:
            x1 = x1.unsqueeze(-1)
        x1 = self.relu(self.embedding1(x1))
        x1 = self.transformer1(x1)
        x1 = x1.reshape(x1.size(0), -1)
        return self.fc(self.dropout(x1))


class TimeSeriesDataset(Dataset):
    """A true time-series-only dataset interface; no Aout tensor is retained."""

    def __init__(self, visit_x: np.ndarray, y: np.ndarray):
        if len(visit_x) != len(y):
            raise ValueError("Time-series and label sample counts are not aligned.")
        self.visit_x = visit_x.astype("float32")
        self.y = y.astype("float32")

    def __len__(self) -> int:
        return len(self.y)

    def __getitem__(self, idx: int):
        return self.visit_x[idx], self.y[idx]


def unwrap_model(model: nn.Module) -> nn.Module:
    return model.module if isinstance(model, nn.DataParallel) else model


def count_trainable_parameters(model: nn.Module) -> int:
    return sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)

In [ ]:
# Cell 5 - common loss evaluation and corrected binary/multiclass metrics
def evaluate_loss(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    device: torch.device,
) -> float:
    """Sample-weighted mean BCE loss. Does not compute or inspect TEST metrics."""
    model.eval()
    total_loss = 0.0
    total_samples = 0
    with torch.no_grad():
        for x1, y in loader:
            x1 = x1.to(device=device, dtype=torch.float32, non_blocking=True)
            y = y.to(device=device, dtype=torch.float32, non_blocking=True)
            logits = model(x1)
            loss = criterion(logits, y)
            batch_n = int(y.size(0))
            total_loss += float(loss.item()) * batch_n
            total_samples += batch_n
    if total_samples == 0:
        raise ValueError("Cannot evaluate an empty loader.")
    return total_loss / total_samples


def macro_ovr_auc_from_columns(y_true_columns: np.ndarray, probabilities: np.ndarray) -> float:
    """Average one-vs-rest AUC over classes present as both positive and negative."""
    class_aucs = []
    for column in range(y_true_columns.shape[1]):
        y_column = y_true_columns[:, column]
        if np.unique(y_column).size < 2:
            continue
        class_aucs.append(roc_auc_score(y_column, probabilities[:, column]))
    return float(np.mean(class_aucs)) if class_aucs else float("nan")


def corrected_classification_metrics(
    logits: np.ndarray,
    labels: np.ndarray,
) -> tuple[float, float]:
    """
    Metrics identical to Two-Tower V2.

    Binary label_binarize output is [N,1], so sigmoid/threshold is mandatory.
    Multiclass prediction uses argmax while AUC uses sigmoid OVR columns.
    """
    logits = np.asarray(logits)
    labels = np.asarray(labels)
    if logits.ndim == 1:
        logits = logits[:, None]
    if labels.ndim == 1:
        labels = labels[:, None]
    if logits.shape != labels.shape:
        raise ValueError(f"Logit/label shape mismatch: {logits.shape} vs {labels.shape}")

    probabilities = 1.0 / (1.0 + np.exp(-np.clip(logits, -50.0, 50.0)))
    if logits.shape[1] == 1:
        y_true = labels[:, 0].astype(np.int64)
        y_pred = (probabilities[:, 0] >= 0.5).astype(np.int64)
        accuracy = accuracy_score(y_true, y_pred)
        auc = (
            roc_auc_score(y_true, probabilities[:, 0])
            if np.unique(y_true).size == 2
            else float("nan")
        )
    else:
        y_true = np.argmax(labels, axis=1)
        y_pred = np.argmax(logits, axis=1)
        accuracy = accuracy_score(y_true, y_pred)
        auc = macro_ovr_auc_from_columns(labels, probabilities)
    return float(auc), float(accuracy)


def evaluate_test_once(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    device: torch.device,
) -> tuple[float, float, float, int]:
    model.eval()
    logits_parts, label_parts = [], []
    total_loss = 0.0
    total_samples = 0
    with torch.no_grad():
        for x1, y in loader:
            x1 = x1.to(device=device, dtype=torch.float32, non_blocking=True)
            y_device = y.to(device=device, dtype=torch.float32, non_blocking=True)
            logits = model(x1)
            loss = criterion(logits, y_device)
            batch_n = int(y.size(0))
            total_loss += float(loss.item()) * batch_n
            total_samples += batch_n
            logits_parts.append(logits.detach().cpu().numpy())
            label_parts.append(y.numpy())

    if total_samples == 0:
        raise ValueError("Cannot evaluate an empty TEST loader.")
    logits_np = np.concatenate(logits_parts, axis=0)
    labels_np = np.concatenate(label_parts, axis=0)
    test_auc, test_acc = corrected_classification_metrics(logits_np, labels_np)
    test_loss = total_loss / total_samples
    return test_auc, test_acc, float(test_loss), int(total_samples)

In [ ]:
# Cell 6 - fast internal self-checks (no UCR files required)
def run_internal_self_checks() -> None:
    """Catch metric and architecture regressions before a long GPU run."""
    binary_logits = np.asarray([[-2.0], [2.0], [1.0], [-1.0]], dtype=np.float32)
    binary_labels = np.asarray([[0.0], [1.0], [0.0], [1.0]], dtype=np.float32)
    binary_auc, binary_acc = corrected_classification_metrics(binary_logits, binary_labels)
    if not np.isclose(binary_acc, 0.5):
        raise AssertionError(f"Binary metric self-check failed: ACC={binary_acc}")

    multiclass_logits = np.asarray(
        [[4.0, -1.0, -2.0], [-1.0, 3.0, 0.0], [0.0, -1.0, 3.0]],
        dtype=np.float32,
    )
    multiclass_labels = np.eye(3, dtype=np.float32)
    _, multiclass_acc = corrected_classification_metrics(
        multiclass_logits, multiclass_labels
    )
    if not np.isclose(multiclass_acc, 1.0):
        raise AssertionError("Multiclass metric self-check failed.")

    cpu_model = TransformerOnly(
        input_dim1=1,
        hidden_dim1=16,
        num_heads=2,
        num_layers=2,
        num_classes=1,
        seq_len1=8,
    ).cpu()
    forbidden_modules = {"embedding2", "transformer2", "final_transformer"}
    present_modules = set(dict(cpu_model.named_modules()))
    leaked_modules = sorted(forbidden_modules.intersection(present_modules))
    if leaked_modules:
        raise AssertionError(f"Removed Two-Tower modules still present: {leaked_modules}")

    cpu_model.eval()
    with torch.no_grad():
        output = cpu_model(torch.randn(2, 8, 1))
    if tuple(output.shape) != (2, 1):
        raise AssertionError(f"Model forward shape is wrong: {tuple(output.shape)}")

    print(
        "INTERNAL SELF-CHECKS PASSED | "
        f"binary ACC test={binary_acc:.2f} (expected 0.50, not forced to 1.00) | "
        f"binary AUC test={binary_auc:.2f} | model output={tuple(output.shape)} | "
        "uses_aout=False | fusion_removed=True"
    )


run_internal_self_checks()

In [ ]:
# Cell 7 - one complete dataset run
def run_one_dataset_v2(
    dataset_dir: Path,
    dataset_name: str,
    output_paths: dict[str, Path],
    device: torch.device = DEVICE,
    batch_size: int = BATCH_SIZE,
    num_epochs: int = NUM_EPOCHS,
    lr: float = LEARNING_RATE,
    val_fraction: float = VAL_FRACTION,
    save_curve: bool = SAVE_INDIVIDUAL_CURVES,
) -> tuple[dict, pd.DataFrame]:
    set_global_seed(SEED)
    if device.type == "cuda":
        torch.cuda.empty_cache()

    # Aout markers are checked only so discovery matches the completed Two-Tower
    # V2 dataset universe. Their values are not read or passed to this model.
    required_paths = {
        "train_tsv": dataset_dir / f"{dataset_name}_TRAIN_cleaned.tsv",
        "test_tsv": dataset_dir / f"{dataset_name}_TEST_cleaned.tsv",
        "aout_train_marker": dataset_dir / f"{dataset_name}_Aout_train_k2.csv",
        "aout_test_marker": dataset_dir / f"{dataset_name}_Aout_test_k2.csv",
    }
    missing = [str(path) for path in required_paths.values() if not path.exists()]
    if missing:
        raise FileNotFoundError("Missing required files: " + "; ".join(missing))

    # Official TRAIN and TEST are never concatenated.
    tsv_train = pd.read_csv(required_paths["train_tsv"], sep="\t", header=None)
    tsv_test = pd.read_csv(required_paths["test_tsv"], sep="\t", header=None)

    y_train_raw = tsv_train.iloc[:, 0].to_numpy()
    y_test_raw = tsv_test.iloc[:, 0].to_numpy()
    visit_train_raw = tsv_train.iloc[:, 1:].to_numpy(dtype="float32")
    visit_test_raw = tsv_test.iloc[:, 1:].to_numpy(dtype="float32")

    # Determine sequence length only from official TRAIN, exactly as Two-Tower V2.
    temporary_train, _, _ = clean_and_pad_timeseries(
        visit_train_raw,
        min_len=MIN_EFFECTIVE_LENGTH,
        fixed_len=None,
    )
    train_sequence_length = int(temporary_train.shape[1])

    visit_train_clean, keep_train, _ = clean_and_pad_timeseries(
        visit_train_raw,
        min_len=MIN_EFFECTIVE_LENGTH,
        fixed_len=train_sequence_length,
    )
    visit_test_clean, keep_test, _ = clean_and_pad_timeseries(
        visit_test_raw,
        min_len=MIN_EFFECTIVE_LENGTH,
        fixed_len=train_sequence_length,
    )
    y_train = y_train_raw[keep_train]
    y_test = y_test_raw[keep_test]

    classes = np.sort(np.unique(y_train))
    if len(classes) < 2:
        raise ValueError("Training split contains fewer than two classes.")
    unseen_test = np.setdiff1d(np.unique(y_test), classes)
    if len(unseen_test):
        raise ValueError(f"Official TEST contains labels absent from TRAIN: {unseen_test}")

    y_train_encoded = label_binarize(y_train, classes=classes).astype("float32")
    y_test_encoded = label_binarize(y_test, classes=classes).astype("float32")
    if y_train_encoded.ndim == 1:
        y_train_encoded = y_train_encoded[:, None]
        y_test_encoded = y_test_encoded[:, None]

    train_idx, val_idx, split_path = make_class_safe_split(
        labels=y_train,
        dataset_name=dataset_name,
        split_dir=output_paths["shared_splits"],
        val_fraction=val_fraction,
        seed=SEED,
    )

    visit_train = visit_train_clean[:, :, None].astype("float32")
    visit_test = visit_test_clean[:, :, None].astype("float32")

    train_dataset = TimeSeriesDataset(
        visit_train[train_idx],
        y_train_encoded[train_idx],
    )
    val_dataset = TimeSeriesDataset(
        visit_train[val_idx],
        y_train_encoded[val_idx],
    )
    test_dataset = TimeSeriesDataset(visit_test, y_test_encoded)

    generator = torch.Generator(device="cpu").manual_seed(SEED)
    pin_memory = device.type == "cuda"
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        generator=generator,
        pin_memory=pin_memory,
        num_workers=NUM_WORKERS,
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        pin_memory=pin_memory,
        num_workers=NUM_WORKERS,
    )
    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        pin_memory=pin_memory,
        num_workers=NUM_WORKERS,
    )

    num_outputs = int(y_train_encoded.shape[1])
    model = TransformerOnly(
        input_dim1=int(visit_train.shape[2]),
        hidden_dim1=16,
        num_heads=2,
        num_layers=2,
        num_classes=num_outputs,
        seq_len1=int(visit_train.shape[1]),
    ).to(device)

    parameter_count = count_trainable_parameters(model)
    print(
        f"[{dataset_name}] model={model.__class__.__name__} | "
        f"uses_aout=False | fusion_removed=True | outputs={num_outputs} | "
        f"classes={len(classes)} | params={parameter_count:,} | train/val/test="
        f"{len(train_dataset)}/{len(val_dataset)}/{len(test_dataset)}"
    )

    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)
    scheduler = CosineAnnealingLR(optimizer, T_max=T_MAX, eta_min=ETA_MIN)

    checkpoint_path = output_paths["checkpoints"] / f"{dataset_name}_best.pt"
    best_val_loss = float("inf")
    best_epoch = 0
    history_rows = []

    for epoch in range(1, num_epochs + 1):
        model.train()
        train_loss_sum = 0.0
        train_samples = 0
        for x1, y in train_loader:
            x1 = x1.to(device=device, dtype=torch.float32, non_blocking=True)
            y = y.to(device=device, dtype=torch.float32, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            logits = model(x1)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()

            batch_n = int(y.size(0))
            train_loss_sum += float(loss.item()) * batch_n
            train_samples += batch_n

        train_loss = train_loss_sum / train_samples
        val_loss = evaluate_loss(model, val_loader, criterion, device)
        current_lr = float(optimizer.param_groups[0]["lr"])
        history_rows.append(
            {
                "dataset": dataset_name,
                "model": MODEL_NAME,
                "seed": SEED,
                "epoch": epoch,
                "train_loss": train_loss,
                "val_loss": val_loss,
                "learning_rate": current_lr,
            }
        )

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_epoch = epoch
            checkpoint = {
                "experiment_version": EXPERIMENT_VERSION,
                "dataset": dataset_name,
                "model_name": MODEL_NAME,
                "model_class": unwrap_model(model).__class__.__name__,
                "state_dict": copy.deepcopy(unwrap_model(model).state_dict()),
                "best_epoch": best_epoch,
                "best_val_loss": best_val_loss,
                "classes": classes.tolist(),
                "parameter_count": parameter_count,
                "train_sequence_length": train_sequence_length,
                "uses_aout": False,
                "fusion_removed": True,
                "split_path": str(split_path),
            }
            torch.save(checkpoint, checkpoint_path)

        scheduler.step()

    history_df = pd.DataFrame(history_rows)
    atomic_write_csv(
        history_df,
        output_paths["dataset_histories"] / f"{dataset_name}.csv",
    )

    # The checkpoint was created by this notebook and is therefore trusted.
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
    unwrap_model(model).load_state_dict(checkpoint["state_dict"])
    test_auc, test_acc, test_loss, test_n = evaluate_test_once(
        model, test_loader, criterion, device
    )

    if save_curve:
        figure, axis = plt.subplots(figsize=(6.2, 4.0))
        axis.plot(history_df["epoch"], history_df["train_loss"], label="Train BCE")
        axis.plot(history_df["epoch"], history_df["val_loss"], label="Validation BCE")
        axis.axvline(best_epoch, color="black", linestyle="--", linewidth=1, label="Best epoch")
        axis.set(
            title=f"{dataset_name} - Transformer-only V2",
            xlabel="Epoch",
            ylabel="BCE loss",
        )
        axis.grid(alpha=0.25)
        axis.legend()
        figure.tight_layout()
        figure.savefig(output_paths["curves"] / f"{dataset_name}_loss.png", dpi=180)
        plt.close(figure)

    result = {
        "dataset": dataset_name,
        "model": MODEL_NAME,
        "seed": SEED,
        "test_auc": test_auc,
        "test_acc": test_acc,
        "test_loss": test_loss,
        "n_samples": test_n,
        "num_classes": int(len(classes)),
        "num_outputs": num_outputs,
        "best_epoch": int(best_epoch),
        "best_val_loss": float(best_val_loss),
        "train_samples": int(len(train_dataset)),
        "val_samples": int(len(val_dataset)),
        "parameter_count": int(parameter_count),
        "uses_aout": False,
        "fusion_removed": True,
        "status": "completed",
    }
    print(
        f"[{dataset_name}] TEST loss={test_loss:.4f} | AUC={test_auc:.4f} | "
        f"ACC={test_acc:.4f} | best_epoch={best_epoch} | n={test_n}"
    )
    return result, history_df

In [ ]:
# Cell 8 - discovery, resumable all-dataset runner, summaries, and persistence
def discover_datasets(root: Path) -> list[str]:
    # Require the same four marker files as Two-Tower V2 so both models run
    # on exactly the same dataset universe; Aout values are never loaded here.
    if not root.exists():
        raise FileNotFoundError(f"DATA_ROOT does not exist: {root}")
    names = []
    for subdir in sorted(path for path in root.iterdir() if path.is_dir()):
        name = subdir.name
        required = [
            subdir / f"{name}_TRAIN_cleaned.tsv",
            subdir / f"{name}_TEST_cleaned.tsv",
            subdir / f"{name}_Aout_train_k2.csv",
            subdir / f"{name}_Aout_test_k2.csv",
        ]
        if all(path.exists() for path in required):
            names.append(name)
    return names


def summarize_results(results_df: pd.DataFrame) -> dict:
    valid_auc = results_df.dropna(subset=["test_auc"])
    total_n = float(results_df["n_samples"].sum())
    auc_weight_n = float(valid_auc["n_samples"].sum())
    return {
        "datasets_completed": int(len(results_df)),
        "simple_mean_auc": float(valid_auc["test_auc"].mean()),
        "simple_mean_acc": float(results_df["test_acc"].mean()),
        "simple_mean_loss": float(results_df["test_loss"].mean()),
        "weighted_auc": float(
            (valid_auc["test_auc"] * valid_auc["n_samples"]).sum() / auc_weight_n
        ),
        "weighted_acc": float(
            (results_df["test_acc"] * results_df["n_samples"]).sum() / total_n
        ),
    }


def run_datasets_v2(
    root: Path,
    selected: list[str] | None = None,
    debug: bool = False,
    num_epochs: int = NUM_EPOCHS,
    resume: bool = RESUME_COMPLETED_DATASETS,
) -> tuple[pd.DataFrame, dict]:
    base_dir = model_output_dir(debug=debug)
    output_paths = ensure_output_tree(base_dir)
    final_results_path = base_dir / "final_results.csv"
    histories_path = base_dir / "histories.csv"
    failed_path = base_dir / "failed_datasets.csv"
    config_path = base_dir / "run_config.json"
    summary_path = base_dir / "summary.json"

    config = base_run_config(debug=debug, epochs=num_epochs)
    atomic_write_json(config, config_path)

    available = discover_datasets(root)
    if selected is None:
        dataset_names = available
    else:
        missing = sorted(set(selected) - set(available))
        if missing:
            raise FileNotFoundError(f"Selected datasets not discoverable: {missing}")
        dataset_names = [name for name in selected if name in available]

    completed = set()
    if resume and final_results_path.exists():
        existing = pd.read_csv(final_results_path)
        if "dataset" in existing.columns:
            completed = set(existing.loc[existing["status"] == "completed", "dataset"])

    print(f"Output directory: {base_dir}")
    print(f"Datasets requested: {len(dataset_names)}")
    print(f"Already completed and skipped: {len(completed.intersection(dataset_names))}")

    for position, dataset_name in enumerate(dataset_names, start=1):
        if dataset_name in completed:
            print(f"[{position}/{len(dataset_names)}] Skip completed: {dataset_name}")
            continue
        print(f"\n[{position}/{len(dataset_names)}] Start: {dataset_name}")
        try:
            result, history_df = run_one_dataset_v2(
                dataset_dir=root / dataset_name,
                dataset_name=dataset_name,
                output_paths=output_paths,
                device=DEVICE,
                batch_size=BATCH_SIZE,
                num_epochs=num_epochs,
                lr=LEARNING_RATE,
                val_fraction=VAL_FRACTION,
                save_curve=SAVE_INDIVIDUAL_CURVES,
            )
            append_or_replace_dataset_rows(
                final_results_path,
                pd.DataFrame([result]),
                dataset_name,
            )
            append_or_replace_dataset_rows(
                histories_path,
                history_df,
                dataset_name,
            )

            # Remove an earlier failure record if a retry succeeds.
            if failed_path.exists():
                failed_df = pd.read_csv(failed_path)
                failed_df = failed_df[failed_df["dataset"] != dataset_name]
                atomic_write_csv(failed_df, failed_path)
        except Exception as error:
            failure = pd.DataFrame(
                [
                    {
                        "dataset": dataset_name,
                        "model": MODEL_NAME,
                        "seed": SEED,
                        "error_type": type(error).__name__,
                        "error_message": str(error),
                        "traceback": traceback.format_exc(),
                        "recorded_utc": datetime.now(timezone.utc).isoformat(),
                    }
                ]
            )
            append_or_replace_dataset_rows(failed_path, failure, dataset_name)
            print(f"FAILED {dataset_name}: {type(error).__name__}: {error}")

    if not final_results_path.exists():
        raise RuntimeError("No dataset completed successfully.")

    final_results = pd.read_csv(final_results_path).sort_values("dataset").reset_index(drop=True)
    requested_results = final_results[final_results["dataset"].isin(dataset_names)].copy()
    summary = summarize_results(requested_results)
    summary.update(
        {
            "model": MODEL_NAME,
            "seed": SEED,
            "debug": bool(debug),
            "datasets_requested": int(len(dataset_names)),
            "generated_utc": datetime.now(timezone.utc).isoformat(),
        }
    )
    atomic_write_json(summary, summary_path)

    print("\n========== V2 SUMMARY (OFFICIAL TEST, EVALUATED ONCE) ==========")
    print(requested_results[["dataset", "test_auc", "test_acc", "test_loss", "n_samples"]].to_string(index=False))
    print(
        f"\nSimple mean: AUC={summary['simple_mean_auc']:.4f}, "
        f"ACC={summary['simple_mean_acc']:.4f}, LOSS={summary['simple_mean_loss']:.4f}"
    )
    print(
        f"Weighted: AUC={summary['weighted_auc']:.4f}, "
        f"ACC={summary['weighted_acc']:.4f}"
    )
    return requested_results, summary

## Smoke test

Leave `RUN_SMOKE_TEST = True` in Cell 1 and restart/run all. Results go only to
`final_runs_v2/_debug/transformer_only/seed_42`, so they cannot be mistaken for
final results or overwrite the completed Two-Tower V2 run.

In [ ]:
# Cell 9 - safe three-epoch smoke test (enabled by default)
if RUN_SMOKE_TEST:
    smoke_results, smoke_summary = run_datasets_v2(
        root=DATA_ROOT,
        selected=SMOKE_DATASETS,
        debug=True,
        num_epochs=SMOKE_EPOCHS,
        resume=False,
    )
else:
    print("Smoke test disabled. Set RUN_SMOKE_TEST=True in Cell 1 to run it.")

## Final 125-dataset Transformer-only run

After the smoke test succeeds:

1. set `RUN_SMOKE_TEST = False`;
2. set `RUN_FULL_EXPERIMENT = True`;
3. restart the kernel and run all cells.

The run updates `final_results.csv` and `histories.csv` after every completed
dataset. With resume enabled, rerunning the notebook skips completed datasets.
The shared split files created by Two-Tower V2 are reused and validated.

In [ ]:
# Cell 10 - final full run (disabled by default)
if RUN_FULL_EXPERIMENT:
    full_results, full_summary = run_datasets_v2(
        root=DATA_ROOT,
        selected=None,
        debug=False,
        num_epochs=NUM_EPOCHS,
        resume=RESUME_COMPLETED_DATASETS,
    )
else:
    print("Full run disabled. Set RUN_FULL_EXPERIMENT=True in Cell 1 after smoke testing.")


## Files produced by the final run

```text
final_runs_v2/
└── transformer_only/
    └── seed_42/
        ├── final_results.csv
        ├── histories.csv
        ├── failed_datasets.csv          # created only if a dataset fails
        ├── run_config.json
        ├── summary.json
        ├── checkpoints/
        ├── curves/
        └── dataset_histories/
```

`histories.csv` is directly comparable with the Two-Tower V2 histories for
revised validation-loss figures. `final_results.csv` is the direct source for
the Transformer-only row in Table 2 and later per-dataset ablation-delta figures.